In [5]:
import pandas as pd
from pathlib import Path

# -------------------------------
# Load dataset
# -------------------------------

DATA_PATH = r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv"
df = pd.read_csv(DATA_PATH)

print("Original dataset shape:", df.shape)

# -------------------------------
# Create new label
# -------------------------------

df["Ad_shown"] = df["treatment"].astype(int)

# -------------------------------
# Split exposed vs not exposed
# -------------------------------

df_exposed = df[df["Ad_shown"] == 1].reset_index(drop=True)
df_not_exposed = df[df["Ad_shown"] == 0].reset_index(drop=True)

print("\nDataset A — AD EXPOSED:", df_exposed.shape)
print("Dataset B — NOT EXPOSED:", df_not_exposed.shape)

# -------------------------------
# Save both datasets
# -------------------------------

SAVE_DIR = r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\Uplift\A_vs_B"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

PATH_EXPOSED = str(Path(SAVE_DIR)/"dataset_A_exposed.csv")
PATH_NOT_EXPOSED = str(Path(SAVE_DIR)/"dataset_B_not_exposed.csv")

df_exposed.to_csv(PATH_EXPOSED, index=False)
df_not_exposed.to_csv(PATH_NOT_EXPOSED, index=False)

print("\nSaved:")
print("A (exposed):", PATH_EXPOSED)
print("B (not exposed):", PATH_NOT_EXPOSED)
print("\nNext Step: Train Purchase/Churn/CLV models separately on A and B.")


Original dataset shape: (65911, 137)

Dataset A — AD EXPOSED: (27301, 138)
Dataset B — NOT EXPOSED: (38610, 138)

Saved:
A (exposed): C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\Uplift\A_vs_B\dataset_A_exposed.csv
B (not exposed): C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\Uplift\A_vs_B\dataset_B_not_exposed.csv

Next Step: Train Purchase/Churn/CLV models separately on A and B.


In [10]:
# Uplift Analysis Notebook (Jupyter .py cells)
# This script is intended to be used as a Jupyter notebook ("# %%" cell markers).
# It performs simple uplift analysis (A vs B) using the CLV master dataset.

# %%
# Title: Simple Uplift Analysis
# Author: Generated by Assistant
# Description: Loads CLV dataset, creates Ad_shown from 'treatment', splits exposed vs not exposed,
# computes uplift for purchase, CLV (spend), churn and engagement metrics, runs statistical tests,
# and saves summary CSVs + plots for dashboard ingestion.

# %%
# Requirements:
# pip install pandas numpy matplotlib seaborn scipy statsmodels

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

# Notebook settings
pd.options.display.max_columns = 200
sns.set(style="whitegrid")

# %%
# CONFIG - update paths if needed
DATA_PATH = r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv"
OUT_DIR = Path(r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\Uplift\simple_uplift_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TREAT_COL = "treatment"  # existing column to use as Ad_shown
PERSON_COL = "person"

# Outcomes to evaluate (auto-detect availability)
PURCHASE_COLS = ["purchase_next_1d", "purchase_next_7d", "purchase_next_30d_true", "purchase_next_90d_true"]
CLV_COLS = ["future_30d_spend_true", "future_90d_spend_true"]
CHURN_COL = "churn_30d"

# Engagement metrics (optional)
ENGAGEMENT_COLS = ["views_last_7d", "session_count_7d", "ad_exposures_7d"]

# %%
# Load data
print("Loading:", DATA_PATH)
df = pd.read_csv(DATA_PATH)
print("Rows, cols:", df.shape)

# sanity: ensure treatment present
if TREAT_COL not in df.columns:
    raise KeyError(f"Treatment column '{TREAT_COL}' not found in dataset.")

# create Ad_shown column (duplicate of treatment for clarity)
df["Ad_shown"] = df[TREAT_COL].astype(int)

# %%
# Build A/B groups
df_exposed = df[df["Ad_shown"] == 1].reset_index(drop=True)
df_control = df[df["Ad_shown"] == 0].reset_index(drop=True)

print("Exposed rows:", len(df_exposed))
print("Control rows:", len(df_control))

# quick sanity rates
print("Exposed purchase (30d) rate:", df_exposed["purchase_next_30d"].mean() if "purchase_next_30d" in df_exposed.columns else None)

# %%
# Utility: proportion z-test for binary outcomes
def uplift_proportion(outcome_col, alpha=0.05):
    """Returns dict with group means, uplift, z-test p-value and CI"""
    a = df_exposed[outcome_col].dropna().astype(int)
    b = df_control[outcome_col].dropna().astype(int)
    n1 = len(a); n2 = len(b)
    p1 = a.mean(); p2 = b.mean()
    uplift = p1 - p2
    # z-test
    count = np.array([a.sum(), b.sum()])
    nobs = np.array([n1, n2])
    stat, pval = proportions_ztest(count, nobs)
    # confidence interval for difference using normal approx
    se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    z = stats.norm.ppf(1 - alpha/2)
    ci_low = uplift - z*se
    ci_high = uplift + z*se
    return {"n_exposed": n1, "n_control": n2, "p_exposed": p1, "p_control": p2,
            "uplift_abs": uplift, "uplift_rel_pct": (uplift / p2 * 100) if p2>0 else np.nan,
            "z_stat": stat, "p_value": pval, "ci": (ci_low, ci_high)}

# %%
# Utility: difference of means t-test for continuous outcomes
def uplift_mean(outcome_col, alpha=0.05):
    a = df_exposed[outcome_col].dropna().astype(float)
    b = df_control[outcome_col].dropna().astype(float)
    n1 = len(a); n2 = len(b)
    mu1 = a.mean(); mu2 = b.mean()
    uplift = mu1 - mu2
    # Welch's t-test
    tstat, pval = stats.ttest_ind(a, b, equal_var=False, nan_policy='omit')
    # 95% CI for difference using t distribution
    se = np.sqrt(a.var(ddof=1)/n1 + b.var(ddof=1)/n2)
    dfree = (a.var(ddof=1)/n1 + b.var(ddof=1)/n2)**2 / ((a.var(ddof=1)**2 / (n1**2 * (n1-1))) + (b.var(ddof=1)**2 / (n2**2 * (n2-1))))
    t_crit = stats.t.ppf(1 - alpha/2, df=dfree)
    ci_low = uplift - t_crit*se
    ci_high = uplift + t_crit*se
    return {"n_exposed": n1, "n_control": n2, "mean_exposed": mu1, "mean_control": mu2,
            "uplift_abs": uplift, "uplift_rel_pct": (uplift / mu2 * 100) if mu2>0 else np.nan,
            "t_stat": tstat, "p_value": pval, "ci": (ci_low, ci_high)}

# %%
# Compute uplift for all available outcome columns
results = {"purchase": {}, "clv": {}, "churn": None, "engagement": {}}

# purchases
for col in PURCHASE_COLS:
    if col in df.columns:
        print(f"Processing purchase column: {col}")
        res = uplift_proportion(col)
        results["purchase"][col] = res

# CLV (continuous)
for col in CLV_COLS:
    if col in df.columns:
        print(f"Processing CLV column: {col}")
        res = uplift_mean(col)
        results["clv"][col] = res

# churn
if CHURN_COL in df.columns:
    print("Processing churn column:", CHURN_COL)
    results["churn"] = uplift_proportion(CHURN_COL)

# engagement
for col in ENGAGEMENT_COLS:
    if col in df.columns:
        print("Processing engagement column:", col)
        results["engagement"][col] = uplift_mean(col)

# Save raw results to CSV-friendly dataframe
rows = []
# purchases
for col, v in results["purchase"].items():
    rows.append({"metric": col, "type": "binary", **v})
# clv
for col, v in results["clv"].items():
    rows.append({"metric": col, "type": "continuous", **v})
# churn
if results["churn"]:
    rows.append({"metric": CHURN_COL, "type": "binary", **results["churn"]})
# engagement
for col, v in results["engagement"].items():
    rows.append({"metric": col, "type": "continuous", **v})

summary_df = pd.DataFrame(rows)
summary_csv = OUT_DIR / "uplift_summary_table.csv"
summary_df.to_csv(summary_csv, index=False)
print("Saved uplift summary to:", summary_csv)

# %%
# Quick plotting helpers

def plot_binary_uplift(col, outdir=OUT_DIR):
    data = {
        'Group': ['Exposed', 'Control'],
        'Rate': [df_exposed[col].mean(), df_control[col].mean()],
        'Count': [len(df_exposed[col].dropna()), len(df_control[col].dropna())]
    }
    fig, ax = plt.subplots(figsize=(6,4))
    sns.barplot(x='Group', y='Rate', data=pd.DataFrame(data), ax=ax)
    ax.set_title(f'Uplift - {col} (rates)')
    for i, v in enumerate(data['Rate']):
        ax.text(i, v + 0.005, f"{v:.3f}", ha='center')
    plt.tight_layout()
    p = outdir / f"uplift_{col}_rate.png"
    fig.savefig(p)
    plt.close(fig)
    return p


def plot_mean_uplift(col, outdir=OUT_DIR):
    mu1 = df_exposed[col].mean(); mu2 = df_control[col].mean()
    fig, ax = plt.subplots(figsize=(6,4))
    sns.barplot(x=['Exposed','Control'], y=[mu1, mu2], ax=ax)
    ax.set_title(f'Uplift - {col} (means)')
    for i, v in enumerate([mu1, mu2]):
        ax.text(i, v + (0.01 * abs(v) if v!=0 else 0.01), f"{v:.2f}", ha='center')
    plt.tight_layout()
    p = outdir / f"uplift_{col}_mean.png"
    fig.savefig(p)
    plt.close(fig)
    return p

# %%
# Produce plots for each metric
plots = []
for col in results['purchase'].keys():
    p = plot_binary_uplift(col)
    plots.append(p)
for col in results['clv'].keys():
    p = plot_mean_uplift(col)
    plots.append(p)
if results['churn']:
    p = plot_binary_uplift(CHURN_COL)
    plots.append(p)
for col in results['engagement'].keys():
    p = plot_mean_uplift(col)
    plots.append(p)

print("Saved plots:")
for p in plots:
    print(p)

# %%
# Save detailed per-user CSVs for dashboarding: include predictions as just group membership + outcomes
out_user = df[[PERSON_COL, 'Ad_shown'] + [c for c in df.columns if c in (PURCHASE_COLS + CLV_COLS + [CHURN_COL])]].copy()
out_user.to_csv(OUT_DIR / 'uplift_per_user.csv', index=False)
print('Saved per-user uplift CSV to', OUT_DIR / 'uplift_per_user.csv')

# %%
# Summary printing (human readable)
print('\n===== UPLIFT SUMMARY =====')
print(summary_df[['metric','type','n_exposed','n_control','uplift_abs','uplift_rel_pct','p_value']])

# Optional: show top-level KPIs
if 'purchase_next_30d' in df.columns:
    print('\nOverall purchase_next_30d: Exposed', df_exposed['purchase_next_30d'].mean(), 'Control', df_control['purchase_next_30d'].mean())
if 'future_30d_spend_true' in df.columns:
    print('Overall future_30d_spend_true: Exposed', df_exposed['future_30d_spend_true'].mean(), 'Control', df_control['future_30d_spend_true'].mean())

# %%
# End of notebook
print('\nDone. Files saved to', OUT_DIR)


Loading: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv
Rows, cols: (65911, 137)
Exposed rows: 27301
Control rows: 38610
Exposed purchase (30d) rate: 0.3129555693930625
Processing purchase column: purchase_next_1d
Processing purchase column: purchase_next_7d
Processing purchase column: purchase_next_30d_true
Processing purchase column: purchase_next_90d_true
Processing CLV column: future_30d_spend_true
Processing CLV column: future_90d_spend_true
Processing churn column: churn_30d
Processing engagement column: views_last_7d
Processing engagement column: session_count_7d
Processing engagement column: ad_exposures_7d
Saved uplift summary to: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\Uplift\simple_uplift_output\uplift_summary_table.csv
Saved plots:
C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\Uplift\simple_uplift_output\uplift_purchase_next_1d_rate.png
C:\Users\abhin\Desktop\Projects\FinTech Project

In [12]:
# uplift_ml_models.py
# Trains T-Learner, S-Learner, X-Learner for purchase probability (binary) and CLV (continuous).
# Saves per-user ITE predictions and model artifacts.
# Requirements: pandas, numpy, scikit-learn, lightgbm, joblib
# pip install pandas numpy scikit-learn lightgbm joblib

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score
import lightgbm as lgb

# ---------------- CONFIG ----------------
DATA_PATH = r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv"
OUT_DIR = Path(r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\Uplift\ml_uplift_models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

TREAT_COL = "treatment"
PERSON_COL = "person"

# Outcome columns (we'll use these)
PURCHASE_COL = "purchase_next_30d_true"   # binary (we'll use predicted probability)
CLV_COL = "future_30d_spend_true"         # continuous

# Feature columns: auto-detect from dataset (drop ids, times, targets)
DROP_CANDIDATES = [PERSON_COL, "anchor_time", "anchor_time_parsed",
                   PURCHASE_COL, "purchase_next_1d", "purchase_next_7d", "purchase_next_90d",
                   "purchase_next_30d", "purchase_next_90d_true",
                   CLV_COL, "future_90d_spend_true", "true_uplift"]  # add known targets to drop

# LightGBM base model params (tune later if desired)
LGB_CLASS_PARAMS = dict(objective="binary", learning_rate=0.03, n_estimators=1000,
                        num_leaves=127, min_data_in_leaf=20, n_jobs=-1, verbose=-1)
LGB_REG_PARAMS = dict(objective="regression", learning_rate=0.03, n_estimators=1500,
                      num_leaves=127, min_data_in_leaf=20, n_jobs=-1, verbose=-1)

# ---------------- HELPERS ----------------
def rmse(y, yhat):
    return mean_squared_error(y, yhat) ** 0.5

# safe predict proba (return probs for positive class)
def predict_proba_safe(model, X):
    try:
        return model.predict_proba(X)[:,1]
    except Exception:
        # maybe model is sklearn regressor that outputs continuous; convert via logistic-like squashing
        raw = model.predict(X)
        # clip to [0,1]
        return np.clip(raw, 0, 1)

# ---------------- LOAD & PREPROCESS ----------------
print("Loading", DATA_PATH)
df = pd.read_csv(DATA_PATH)
print("Rows, cols:", df.shape)

if TREAT_COL not in df.columns:
    raise KeyError(f"Treatment column '{TREAT_COL}' not found.")

# restrict to rows with non-null outcomes
df = df[~df[CLV_COL].isna()].reset_index(drop=True)
print("After dropping rows w/ missing CLV:", df.shape)

# build features: drop identifiers/timestamps/targets
drop_cols = [c for c in DROP_CANDIDATES if c in df.columns]
X_df = df.drop(columns=drop_cols, errors="ignore").copy()

# detect categorical columns (object/category)
cat_cols = [c for c in X_df.columns if X_df[c].dtype == "object" or X_df[c].dtype.name == "category"]
print("Categorical columns detected:", cat_cols)

# deterministic factorize (save mapping)
cat_mappings = {}
X_fact = X_df.copy()
for c in cat_cols:
    codes, uniques = pd.factorize(X_fact[c].astype(str), sort=False)
    X_fact[c] = codes
    cat_mappings[c] = list(uniques)
joblib.dump(cat_mappings, OUT_DIR / "cat_mappings.joblib")

# impute (median)
imputer = SimpleImputer(strategy="median")
X_num = pd.DataFrame(imputer.fit_transform(X_fact), columns=X_fact.columns).astype(float)
joblib.dump(imputer, OUT_DIR / "imputer.joblib")
print("Saved imputer and cat mappings.")

# features and treatment/outcomes
X = X_num.values
feature_names = X_num.columns.tolist()
T = df[TREAT_COL].astype(int).values
Y_binary = df[PURCHASE_COL].astype(int).values if PURCHASE_COL in df.columns else None
Y_cont = df[CLV_COL].astype(float).values

# split train/test for diagnostics (we will fit on full training set later)
X_train, X_test, T_train, T_test, yb_train, yb_test, yc_train, yc_test = train_test_split(
    X, T, Y_binary, Y_cont, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# ---------------- Propensity model ----------------
print("\nTraining propensity model (logistic)...")
prop_model = lgb.LGBMClassifier(**LGB_CLASS_PARAMS)
prop_model.fit(X_train, T_train)
propensity_full = prop_model.predict_proba(X)[:,1]
propensity_train = prop_model.predict_proba(X_train)[:,1]
propensity_test = prop_model.predict_proba(X_test)[:,1]
joblib.dump(prop_model, OUT_DIR / "propensity_model.joblib")
print("Saved propensity model.")

# ---------------- T-LEARNER ----------------
print("\n--- T-Learner (outcome models separate for treated & control) ---")
# For continuous outcome (CLV)
# Train model on treated units (T==1) to predict Y_cont
mask_tr = T_train == 1
mask_cr = T_train == 0

# train separate regressors on full data splits (we will refit on full data later)
# Using LightGBM regressors
reg_t = lgb.LGBMRegressor(**LGB_REG_PARAMS)
reg_c = lgb.LGBMRegressor(**LGB_REG_PARAMS)

# Fit on train subset where treated / control
reg_t.fit(X_train[mask_tr], yc_train[mask_tr])
reg_c.fit(X_train[mask_cr], yc_train[mask_cr])

# Predict on full X to get μ1 and μ0
mu1_t = reg_t.predict(X)
mu0_c = reg_c.predict(X)

ite_t_learner_cont = mu1_t - mu0_c
# Save models
joblib.dump(reg_t, OUT_DIR / "tlearner_reg_treated.joblib")
joblib.dump(reg_c, OUT_DIR / "tlearner_reg_control.joblib")
print("T-learner reg models saved.")

# For binary outcome (purchase) — use predicted probabilities from separate classifiers
clf_t = lgb.LGBMClassifier(**LGB_CLASS_PARAMS)
clf_c = lgb.LGBMClassifier(**LGB_CLASS_PARAMS)
if Y_binary is not None:
    clf_t.fit(X_train[mask_tr], yb_train[mask_tr])
    clf_c.fit(X_train[mask_cr], yb_train[mask_cr])
    p1 = clf_t.predict_proba(X)[:,1]
    p0 = clf_c.predict_proba(X)[:,1]
    ite_t_learner_bin = p1 - p0
    joblib.dump(clf_t, OUT_DIR / "tlearner_clf_treated.joblib")
    joblib.dump(clf_c, OUT_DIR / "tlearner_clf_control.joblib")
    print("T-learner classifiers saved.")
else:
    ite_t_learner_bin = np.full(len(X), np.nan)

# ---------------- S-LEARNER ----------------
print("\n--- S-Learner (single model with treatment as feature) ---")
# Build X_with_t for full data and for train/test using the previously created splits
X_with_t_full = np.hstack([X, T.reshape(-1,1)])              # full dataset with treatment as feature
X_with_t_train = np.hstack([X_train, T_train.reshape(-1,1)]) # train split with treatment
X_with_t_test  = np.hstack([X_test,  T_test.reshape(-1,1)])  # test split with treatment

# Fit S-learner regressor on TRAIN split (consistent with earlier splits)
s_reg = lgb.LGBMRegressor(**LGB_REG_PARAMS)
s_reg.fit(X_with_t_train, yc_train)  # yc_train was produced by the earlier train_test_split

# To get ITE estimates on full dataset: predict with T=1 and T=0 for all rows
X_t1_full = np.hstack([X, np.ones((X.shape[0],1))])
X_t0_full = np.hstack([X, np.zeros((X.shape[0],1))])
mu_t1_s = s_reg.predict(X_t1_full)
mu_t0_s = s_reg.predict(X_t0_full)
ite_s_learner_cont = mu_t1_s - mu_t0_s
joblib.dump(s_reg, OUT_DIR / "slearner_reg.joblib")
print("S-learner reg saved.")

# S-learner for binary outcome: single classifier producing prob with T as feature
if Y_binary is not None:
    s_clf = lgb.LGBMClassifier(**LGB_CLASS_PARAMS)
    s_clf.fit(X_with_t_train, yb_train)
    p_t1 = s_clf.predict_proba(X_t1_full)[:,1]
    p_t0 = s_clf.predict_proba(X_t0_full)[:,1]
    ite_s_learner_bin = p_t1 - p_t0
    joblib.dump(s_clf, OUT_DIR / "slearner_clf.joblib")
    print("S-learner clf saved.")
else:
    ite_s_learner_bin = np.full(len(X), np.nan)


# ---------------- X-LEARNER ----------------
print("\n--- X-Learner ---")
# Step 1: fit μ0 (control outcome) and μ1 (treated outcome) on all data (we already have reg_c, reg_t for cont; clf_c, clf_t for bin)
# Step 2: compute pseudo outcomes:
#  for treated units: D1_i = y_i - μ0(x_i)
#  for control units: D0_i = μ1(x_i) - y_i
# Step 3: fit models τ1(x) on treated (X1 -> D1), τ0(x) on control (X0 -> D0)
# Step 4: combine predictions using propensity e(x): τ_hat(x) = e(x) * τ0_hat(x) + (1-e(x))*τ1_hat(x)

# Continuous outcome X-learner
# Use the earlier reg_t & reg_c (fitted on train splits). For robustness, re-fit μ models on full data:
mu1_full = lgb.LGBMRegressor(**LGB_REG_PARAMS); mu0_full = lgb.LGBMRegressor(**LGB_REG_PARAMS)
mu1_full.fit(X[T==1], Y_cont[T==1])
mu0_full.fit(X[T==0], Y_cont[T==0])

# Pseudo outcomes
D1 = Y_cont[T==1] - mu0_full.predict(X[T==1])
D0 = mu1_full.predict(X[T==0]) - Y_cont[T==0]

# Fit τ models
tau1_model = lgb.LGBMRegressor(**LGB_REG_PARAMS)
tau0_model = lgb.LGBMRegressor(**LGB_REG_PARAMS)
if sum(T==1) > 20:
    tau1_model.fit(X[T==1], D1)
if sum(T==0) > 20:
    tau0_model.fit(X[T==0], D0)

# Predictions of τ components
tau1_hat = np.zeros(len(X)); tau0_hat = np.zeros(len(X))
if sum(T==1) > 20:
    tau1_hat = tau1_model.predict(X)
if sum(T==0) > 20:
    tau0_hat = tau0_model.predict(X)

# combine using propensity
e_x = propensity_full
ite_xlearner_cont = e_x * tau0_hat + (1 - e_x) * tau1_hat

joblib.dump(mu1_full, OUT_DIR / "x_mu1_reg.joblib")
joblib.dump(mu0_full, OUT_DIR / "x_mu0_reg.joblib")
joblib.dump(tau1_model, OUT_DIR / "x_tau1_reg.joblib")
joblib.dump(tau0_model, OUT_DIR / "x_tau0_reg.joblib")
print("X-learner regression artifacts saved.")

# Binary outcome X-learner (purchase probability)
if Y_binary is not None:
    # fit mu models for probabilities using classifiers trained on full treated/control data
    mu1_clf = lgb.LGBMClassifier(**LGB_CLASS_PARAMS); mu0_clf = lgb.LGBMClassifier(**LGB_CLASS_PARAMS)
    mu1_clf.fit(X[T==1], Y_binary[T==1])
    mu0_clf.fit(X[T==0], Y_binary[T==0])
    # pseudo outcomes
    D1_b = Y_binary[T==1] - mu0_clf.predict_proba(X[T==1])[:,1]
    D0_b = mu1_clf.predict_proba(X[T==0])[:,1] - Y_binary[T==0]
    # fit tau models (regressors on pseudo outcomes)
    tau1_b = lgb.LGBMRegressor(**LGB_REG_PARAMS)
    tau0_b = lgb.LGBMRegressor(**LGB_REG_PARAMS)
    if sum(T==1) > 20:
        tau1_b.fit(X[T==1], D1_b)
    if sum(T==0) > 20:
        tau0_b.fit(X[T==0], D0_b)
    # predict
    tau1_b_hat = tau1_b.predict(X) if sum(T==1) > 20 else np.zeros(len(X))
    tau0_b_hat = tau0_b.predict(X) if sum(T==0) > 20 else np.zeros(len(X))
    ite_xlearner_bin = e_x * tau0_b_hat + (1 - e_x) * tau1_b_hat
    joblib.dump(mu1_clf, OUT_DIR / "x_mu1_clf.joblib")
    joblib.dump(mu0_clf, OUT_DIR / "x_mu0_clf.joblib")
    joblib.dump(tau1_b, OUT_DIR / "x_tau1_bin.joblib")
    joblib.dump(tau0_b, OUT_DIR / "x_tau0_bin.joblib")
else:
    ite_xlearner_bin = np.full(len(X), np.nan)

# ---------------- CONSOLIDATE & SAVE PREDICTIONS ----------------
print("\nSaving per-user ITE predictions (all methods)...")
preds = pd.DataFrame({
    PERSON_COL: df[PERSON_COL].values if PERSON_COL in df.columns else np.arange(len(df)),
    "treatment": T,
    "propensity": propensity_full,
    # T-learner
    "ite_t_reg": ite_t_learner_cont,
    "ite_t_bin": ite_t_learner_bin,
    # S-learner
    "ite_s_reg": ite_s_learner_cont,
    "ite_s_bin": ite_s_learner_bin,
    # X-learner
    "ite_x_reg": ite_xlearner_cont,
    "ite_x_bin": ite_xlearner_bin,
    # baseline predicted outcomes for reference
    "mu1_reg": mu1_t, "mu0_reg": mu0_c
})

OUT_PRED_CSV = OUT_DIR / "per_user_ite_predictions.csv"
preds.to_csv(OUT_PRED_CSV, index=False)
print("Saved ITE predictions to:", OUT_PRED_CSV)

# ---------------- SIMPLE DIAGNOSTICS ----------------
print("\n--- Diagnostics ---")
def att_estimate(ite):
    # Estimated ATT on treated (mean ite among treated)
    return np.mean(ite[T==1])

def ate_estimate(ite):
    return np.mean(ite)

methods = ["ite_t_reg", "ite_s_reg", "ite_x_reg"]
for m in methods:
    print(f"{m} (CLV) ATE: {ate_estimate(preds[m]):.4f}  ATT: {att_estimate(preds[m]):.4f}")

methods_bin = ["ite_t_bin", "ite_s_bin", "ite_x_bin"]
for m in methods_bin:
    print(f"{m} (purchase prob) ATE: {np.nanmean(preds[m]):.6f}  ATT: {np.nanmean(preds[m][T==1]):.6f}")

# Top-decile policy proxy: rank by predicted ITE and compute observed incremental revenue (proxy)
K = int(0.10 * len(df))
print(f"\nTop-decile policy proxy (K={K}) - continuous (CLV) using ite_x_reg:")
top_idx = np.argsort(-preds["ite_x_reg"].values)[:K]
top = preds.iloc[top_idx]
# observed average CLV in top treated vs top control in original df (use actual CLV column)
actual_clv = df[CLV_COL].values
top_person_idx = top.index.values
avg_treated = np.mean(actual_clv[top[(top['treatment']==1)].index]) if any(top['treatment']==1) else np.nan
avg_control = np.mean(actual_clv[top[(top['treatment']==0)].index]) if any(top['treatment']==0) else np.nan
print("Top-decile avg CLV (treated in top):", avg_treated, " (control in top):", avg_control)

# Save models summary
models_meta = {
    "propensity": str(OUT_DIR / "propensity_model.joblib"),
    "t_learner_reg_treated": str(OUT_DIR / "tlearner_reg_treated.joblib"),
    "t_learner_reg_control": str(OUT_DIR / "tlearner_reg_control.joblib"),
    "s_learner_reg": str(OUT_DIR / "slearner_reg.joblib"),
    "x_mu1_reg": str(OUT_DIR / "x_mu1_reg.joblib"),
    "x_mu0_reg": str(OUT_DIR / "x_mu0_reg.joblib"),
}
joblib.dump(models_meta, OUT_DIR / "models_summary.joblib")

print("\nAll done. Models and per-user ITE predictions saved to:", OUT_DIR)


Loading C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\clv_dataset_with_true_spend.csv
Rows, cols: (65911, 137)
After dropping rows w/ missing CLV: (65911, 137)
Categorical columns detected: ['gender', 'became_member_on', 'behavior_segment', 'loyalty_tier', 'recency_bucket']
Saved imputer and cat mappings.

Training propensity model (logistic)...
Saved propensity model.

--- T-Learner (outcome models separate for treated & control) ---
T-learner reg models saved.
T-learner classifiers saved.

--- S-Learner (single model with treatment as feature) ---
S-learner reg saved.
S-learner clf saved.

--- X-Learner ---
X-learner regression artifacts saved.

Saving per-user ITE predictions (all methods)...
Saved ITE predictions to: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\Uplift\ml_uplift_models\per_user_ite_predictions.csv

--- Diagnostics ---
ite_t_reg (CLV) ATE: 0.9683  ATT: -0.7501
ite_s_reg (CLV) ATE: 0.0000  ATT: 0.0000
ite_x_reg (CLV) ATE: 0.7011  